# CMI-RewardBench · From listening to preference accuracy

CMI-RewardBench evaluates models that judge **generated music**. Each example presents a request and two candidate tracks. CMI-RM assigns each candidate a score for **alignment** and a score for **musicality**. We compare the higher-scoring candidate on each dimension with the human preference.

| Dimension | Listening question |
|---|---|
| Alignment | Which candidate follows the request more closely? |
| Musicality | Which candidate works better as music? |

You will listen to a pair whose human preferences differ across the two dimensions, run the official CMI-RM benchmark on four pairs, and trace the reported accuracy back to the pair-level decisions. The saved audio, labels and scores let you follow the lesson directly in the notebook.

[Official CMI-RewardBench repository](https://github.com/Haiwen-Xia/CMI-RewardBench) · [CMI-Pref dataset](https://huggingface.co/datasets/HaiwenXia/cmi-pref)


## 1 · Set up Colab and load the demo pairs

Open this notebook in Colab and select **Runtime → Change runtime type → GPU**. Clone the official repository and install its requirements. The next cell downloads and extracts the test pairs from the public [tutorial dataset](https://huggingface.co/datasets/Leon299/cmi-rewardbench-tutorial).


In [ ]:
!git clone https://github.com/Haiwen-Xia/CMI-RewardBench.git /content/CMI-RewardBench
%cd /content/CMI-RewardBench


Cloning into '/content/CMI-RewardBench'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 144 (delta 47), reused 119 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.91 MiB | 8.24 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/CMI-RewardBench


In [ ]:
%pip install -r requirements.txt
%pip install -r baselines/requirements.txt scipy pyyaml huggingface_hub torchcodec==0.10.0


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 276.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 379.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 365.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 68.3 MB/s  0:00:08
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 110.1 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 229.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 331.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 72.9 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 154.3 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 161.3 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 147.2 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

archive = hf_hub_download(
    "Leon299/cmi-rewardbench-tutorial",
    "CMI_RewardBench_live_demo.zip",
    repo_type="dataset",
    token=False,
)
!unzip -qo "{archive}" "classroom_data/*" -d /content

data = Path("/content/classroom_data")


CMI_RewardBench_live_demo.zip:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

The demo contains four CMI-Pref test pairs. Each illustrates a different combination of conditioning inputs.

| Condition | Inputs in the request |
|---|---|
| T | Text |
| L | Text and lyrics |
| A | Text and reference audio |
| C | Text, lyrics and reference audio |

A reference audio file is part of the request for A and C. Candidate A and candidate B are the two generated tracks being judged. These are separate roles.


In [ ]:
import pandas as pd

test = pd.read_json(data / "cmi_pref_test.jsonl", lines=True)
display(test[["pair_id", "modality", "prompt"]])


,pair_id,modality,prompt
0,demo-T,T,elegant orchestral classical
1,demo-L,L,"Anime Songs for Japanese 80s Magical Girl, roc..."
2,demo-A,A,melodic japanese folk synth-pop
3,demo-C,C,Trap Fusion with Traditional Vietnamese Music:...


## 2 · Listen to one pair and read the human labels

Use the L pair, where the request includes a style description and lyrics. Before reading the labels, decide which candidate follows the prompt more closely. Then make a separate choice based on musicality.


In [ ]:
example = test.iloc[1]
print("Prompt:", example["prompt"])
print("\nLyrics excerpt:")
print("\n".join(example["lyrics"].splitlines()[:9]))


Prompt: Anime Songs for Japanese 80s Magical Girl, rock, female vocal, upbeat, positive, groovy, passionate

Lyrics excerpt:
[Intro]

[Verse1]
I was just a girl, with dreams so sweet
Wishing on a star, a magical feat
A tiny friend, from a distant land
Gave me a gift, a magic wand

[Chorus]


The players use the first 10 seconds, the same window length used in this lab's model run. The benchmark receives the complete request, including the full lyrics. The published human labels refer to the full recordings.


In [ ]:
import librosa
from IPython.display import Audio, display

audio_a, sr = librosa.load(data / "reward" / example["audio-path"], duration=10)
audio_b, sr = librosa.load(data / "reward" / example["audio2"], duration=10)

print("Candidate A")
display(Audio(audio_a, rate=sr))
print("Candidate B")
display(Audio(audio_b, rate=sr))


Candidate A


Candidate B


### Human preferences

`model_a` and `model_b` are the dataset's labels for the two candidates. Read the annotator's explanation and see whether it reflects the criteria you used.


In [ ]:
print("Alignment:", example["preference-alignment"])
print("Musicality:", example["preference-musicality"])
print("Feedback:", example["feedback"])


Alignment: model_a
Musicality: model_b
Feedback: A follows the rock style specified in the prompt, but the vocal is not female. B, on the other hand, uses a female vocal but is not rock—it's closer to jazz. As a result, both have issues with instruction following. In terms of musical quality, B has more balanced audio mixing.


The annotator chose A for alignment because its style is closer to the requested rock sound, even though its vocal also misses the request. They chose B for musicality because its mix sounds more balanced. The two judgments can differ on the same pair.


## 3 · Score the pairs with CMI-RM

Download the official checkpoint and run the repository's benchmark script. CMI-RM scores the two candidates for each request. The pipeline compares their scores to choose A or B on each dimension and automatically evaluates those choices against the human labels.

The lab uses a 10-second window and a batch size of one. The command writes predictions to `results/tutorial/results.jsonl`, summary accuracy to `metrics_summary.csv` in the same directory, and console details to `tutorial_run.log`.

[CMI-RM checkpoint](https://huggingface.co/HaiwenXia/CMI-RM) · [Official inference and evaluation instructions](https://github.com/Haiwen-Xia/CMI-RewardBench#-quick-start)


In [ ]:
# @title
!hf download HaiwenXia/CMI-RM --local-dir baselines/model


Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]Downloading 'config.yaml' to 'baselines/model/.cache/huggingface/download/_ouC4VcfA9-o6nAiMaExMJW7X2A=.cbecf1caed85b96db22af5a20fba1e39d3619e5d.incomplete'

config.yaml: 100% 790/790 [00:00<00:00, 6.31MB/s]
Download complete. Moving file to baselines/model/config.yaml

README.md: 100% 33.0/33.0 [00:00<00:00, 302kB/s]
Download complete. Moving file to baselines/model/README.md

model.safetensors:   0% 0.00/2.80G [00:00<?, ?B/s]

.gitattributes: 1.52kB [00:00, 705kB/s]
Download complete. Moving file to baselines/model/.gitattributes
Fetching 4 files:  25% 1/4 [00:00<00:02,  1.35it/s]
model.safetensors:   0% 225k/2.80G [00:05<19:55:48, 39.1kB/s]
model.safetensors:   5% 144M/2.80G [00:12<03:10, 14.0MB/s]   
model.safetensors:   7% 198M/2.80G [00:12<02:05, 20.8MB/s]
model.safetensors:  13% 355M/2.80G [00:13<00:55, 44.4MB/s]
model.safetensors:  13% 377M/2.80G [00:13<00:51, 47.0MB/s]
model.safetensors:  14% 405M/2.80G [00:13<00:46, 51.8MB/s]
model.sa

In [ ]:
!python inference_benchmark.py \
    -c baselines/model/model.safetensors \
    --dataset_jsonl {data}/cmi_pref_test.jsonl \
    --dataset_root {data}/reward \
    --device cuda:0 \
    --batch_size 1 \
    --max_dur 10 \
    --output_dir results/tutorial > tutorial_run.log


2026-11-08 01:06:13 | INFO | Config:     baselines/model/config.yaml
2026-11-08 01:06:13 | INFO | Checkpoint: baselines/model/model.safetensors
2026-11-08 01:06:13 | INFO | Mode:       final  |  bf16=True
2026-11-08 01:06:20 | INFO | NumExpr defaulting to 2 threads.
I0000 00:00:1794099987.383614    2651 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 100% 847/847 [00:00<00:00, 5.72MB/s]
config.json: 3.13kB [00:00, 15.8MB/s]
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
model.safetensors: 100% 1.33G/1.33G [00:23<00:00, 56.3MB/s]
config.json: 100% 615/615 [00:00<00:00, 3.46MB/s]
mo

## 4 · Inspect the scores and the benchmark result

The displayed results come from a prior CPU run of the official pipeline on these four pairs. Running Section 3 in Colab generates fresh results. Start with the same L pair you heard earlier.


In [ ]:
results = pd.read_json("results/tutorial/results.jsonl", lines=True)
pair = results[results["pair_id"] == example["pair_id"]].iloc[0]

scores = pd.DataFrame({
    "Alignment": [pair["predicted_text-music alignment"], pair["predicted_text-music alignment_b"]],
    "Musicality": [pair["predicted_musicality"], pair["predicted_musicality_b"]],
}, index=["Candidate A", "Candidate B"])
display(scores.round(3))


,Alignment,Musicality
Candidate A,0.322,-0.145
Candidate B,0.206,0.194


### From scores to a pairwise choice

Read each column separately. In this example, A has the higher alignment score, and B has the higher musicality score. Each choice comes from comparing the two candidates for the same prompt and dimension.


In [ ]:
predictions = results[[
    "modality", "preference-alignment", "predicted_preference-alignment",
    "preference-musicality", "predicted_preference-musicality"
]].copy()
predictions.columns = ["Condition", "Human alignment", "CMI-RM alignment",
                       "Human musicality", "CMI-RM musicality"]
display(predictions.replace({"model_a": "A", "model_b": "B"}))


,Condition,Human alignment,CMI-RM alignment,Human musicality,CMI-RM musicality
0,T,B,B,B,B
1,L,A,A,B,B
2,A,B,B,B,B
3,C,A,B,B,B


### From pairwise choices to accuracy

For each dimension, count the pairs where CMI-RM chose the same candidate as the human annotator. Divide by the number of pairs evaluated. The official summary reports the denominator as `n` and the resulting fraction as `ACC`.


In [ ]:
summary = pd.read_csv("results/tutorial/metrics_summary.csv")
display(summary[["dataset", "task", "n", "ACC"]])


,dataset,task,n,ACC
0,CMI-Pref,alignment,4,0.75
1,CMI-Pref,musicality,4,1.00


In the saved run, alignment agrees on three of the four pairs and musicality agrees on all four. The alignment disagreement is in condition C. These four pairs demonstrate how to read the metric. Dataset-level accuracy requires running the full evaluation set.
